In this notebook, we will demonstrate how autoloader handles schema evolution. The following are the steps to demostrate that
- upload day 3 transaction into s3 transaction folder. This file has a new column called `channel`
- upload day 3 customer into s3 customer folder. This file has some update to previous customer records and new customer records 

**Baseline**

In [0]:
%sql
SELECT COUNT(*) FROM dev_kyc.bronze.transactions;   -- 4,226 if the late file is loaded (4,076 + 150)


In [0]:
%sql
SELECT COUNT(*) FROM dev_kyc.bronze.customers;      -- 215


In [0]:
%sql
DESCRIBE TABLE dev_kyc.bronze.transactions;          -- note: no channel column yet

#### S3 Update 
- navigate to S3 and upload day 3 data for customers and transactions
- Run the Pipeline using Run pipeline not Full refresh
- The first run may fail or stop on the trasaction node. By default, when Auto Loader sees a field it hasn’t seen before, it stops the stream, records the new column in its stored schema, and expects a retry. The error will look like an unknown-field exception naming channel. Pipelines often retry automatically, in which case you’ll see a failed attempt followed by a successful one. If it doesn’t, run the pipeline again. This stop-and-retry behavior is the schema evolution mechanism, not a bug.
- On the successful run, transactions writes about 2,040 new rows and customers about 15. Earlier days aren’t reprocessed.

#### Verify


In [0]:
%sql
-- you should see the newly added channel column now
DESCRIBE TABLE dev_kyc.bronze.transactions;


In [0]:
%sql
SELECT COUNT(*) FROM dev_kyc.bronze.transactions;


In [0]:
%sql
SELECT COUNT(*) FROM dev_kyc.bronze.customers;


In [0]:
%sql
-- Old rows have NULL channel, new rows have it
SELECT source_file,
       COUNT(*) AS row_count,
       COUNT(channel) AS rows_with_channel
FROM dev_kyc.bronze.transactions
GROUP BY source_file
ORDER BY source_file;

In [0]:
%sql
-- Values in the new column
SELECT channel, COUNT(*) FROM dev_kyc.bronze.transactions GROUP BY channel;



In [0]:
%sql
SELECT COUNT(*) FROM dev_kyc.bronze.transactions WHERE _rescued_data IS NOT NULL;

In [0]:
%sql
-- Day 3 rows: expect about 15
SELECT *
FROM dev_kyc.bronze.customers
WHERE source_file LIKE '%dt=2026-10-27%'
ORDER BY customer_id;


In [0]:
%sql
-- Customers with more than one version so far (these feed SCD2)
SELECT customer_id, COUNT(*) AS versions
FROM dev_kyc.bronze.customers
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY versions DESC;

In [0]:
%sql
select * from dev_kyc.bronze.customers where customer_id in ('C00057', 'C00085', 'C00136')
order by customer_id